# ☁️ Dia 78 — Filtrando logs de nuvem até o sinal, mais Azure

**Investigador Cyber 90 Dias** · Fase 6 · Rank Jutsu `A`

> Baseado no *Day 78* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-6-cloud-ai-system-investigation/day-78.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Marta tem cinco minutos antes do conselho: *"Não me mostra planilha. Me diz o que é estranho."* Rafa completa: *"E finge que não sabe o IP de ontem."*

Neste notebook você vai:

1. **Medir o palheiro** (500 GCP, 45 AWS, 9 Azure).
2. **Escrever os cinco filtros** como funções Python (o `jq` de cada um vai citado no comentário).
3. **Gerar a tabela antes/depois** calculada.
4. **Quiz: lista curta ou excluído?**
5. **O funil nos logs da Lia** e a lista curta da Marta em Markdown.

🐍 Só biblioteca padrão do Python. Os dados do caso Blue Harbor vêm do GitHub do original; sem rede, o notebook usa a cópia embutida (com o mesmo SHA-256). Os comandos `jq` do README viram Python equivalente aqui. Rode as células de cima para baixo (Shift+Enter).

## 1. Evidência (com custódia) e o palheiro

In [ ]:
# ------------------------------------------------------------
# CARREGADOR DE EVIDÊNCIAS (offline-first, com custódia)
# 1. Tenta baixar cada arquivo do repositório original no GitHub.
# 2. Confere o SHA-256 contra o valor esperado (o hash do upstream).
# 3. Se a rede falhar OU o hash não bater, usa a cópia embutida aqui.
# ------------------------------------------------------------
import base64          # a cópia embutida está em base64 (texto puro)
import gzip            # ...e comprimida com gzip, para o notebook não ficar enorme
import hashlib         # SHA-256: a impressão digital de cada arquivo
import json            # os logs de nuvem são JSON
import urllib.request  # baixar da internet sem instalar nada

BASE_RAW = "https://raw.githubusercontent.com/thiefdroppers/cyber-investigator-skills/main/curriculum/phase-6-cloud-ai-system-investigation/resources/"

# Hash esperado de cada arquivo (calculado na cópia do upstream deste repositório).
HASH_ESPERADO = {
    "case-blueharbor/gcp/audit-activity.json": "a14cf947e072a91439e91f7c4182dbf1730389b15a06dabc357a746c26bb4660",
    "case-blueharbor/gcp/audit-data-access.json": "59ba5aa0925a4d3071ae576dc093b804457ed9e97a29d32398286c47f0d8fd59",
    "case-blueharbor/aws/cloudtrail-lookup-events.json": "820d3aa67653c7e4588192097195a54697f8158d8af4434ba5990a39909f5926",
    "case-blueharbor/aws/cloudtrail-s3-data-events.json": "fb10395c8aa1c666393a9526762be2431b1a4ac3e79976900875a3d29c9f957b",
    "case-blueharbor/azure/activity-log.json": "aff5cb0d7f210c79ec226c08a439eac65ae955507b766dde069e946cb9a86342",
}

# Cópia embutida (plano B offline): o arquivo inteiro, gzip + base64.
_AMOSTRA = {
    "case-blueharbor/gcp/audit-activity.json": "H4sIAAAAAAAC/+2cW2/bOBaA3/srDAMLtEBNU9Rd89Ig8XSDpmkRZ2awMygMSqJlxro4uji2B/Pf9+jm2EmqOpfuSi0fEieWxHPlR1I64l+ver2/4afX6/MwYXF66vatXn9xRZazK1nNVovr/tvyuB955zRgxeE4umJOmgxtP2MzGttRPKAh9dcpd5IhnJgMHT/KXJq5PEVeFHk+owueICcK/kV+pU7Klzxd1y1Dc2n0ma79iObSS33g+3fpelHIyz/vNDMs/0WFHFQKOsp/n0Ve1S60QLN0xkLQiqY8Ck/DabTTfiGZhw5fUH8UUO7nohJnxtzMZ/Egoe8eMg9xGiAPXLXkDqOOE2VhmivUr1r9Z094FPPNruy/trJvtcj1YHHAkwTOy3WA1hZZyhBEJKWhwxIEn3G6Nau4xItpmLLcX2mcsb1DMUuiLHYOCNUmClkyzJIBo0kqDezhVuQwZosoTgdxFobgDKm/FfBP9deXraEBA0PdOjmWEvqWAaDhdcaS9CNLqUtTuh8Uh/rg/9NFqT5f0pTtmF4dHmeLhc+Z+xtE4siDGOdnvy9yYnCc58RgXEfyfmRqBx2Wz49w0lZClR+1gNoh+zm8c3pK0yzJ3VB6t1IVFHUYX7JLHoC3aFC4hGCiDbA5kPAlxhYxLFVChBh/9rfXbKO/7Ul1R/IcNqm1v5XuU5v5yd2eUbhkwgsePOSXrVv3NE7YksV534arzj9dnh6Par3SZiMwGKH+mTdaNPUAlOhGX91k7lSj5pWAUhugFC26zaQd/QWSXghJkiXJFsbIkNXuIgmMIJZqIoJxI5LCK3N2M4tVmnnOz4Ukhw9ctvCj9f+LR5DgqBKQoGzh7nfJw2EUQqySBc27zoNRqmUM825F/QE49nDu7Dk/1xhANK6VvgD/gdzq/yeQSDIJgmEzz9IDaVTqMyj0GSTuvOcVfw4Vw4SWcA9CFNDQHXq3GpdR/jawnuLHr7Epl/vCXJIU6M8WwUjWtYO4VHhgAppMYrbkRda1hE6lKZKJZCw10mlG5dgz9HWiS+ufi04u2IGiOZ1G8Q6fEFuBN332khSqMgh6DfVYjGqfAprSUxp8jnzurF9+knQ4gcYP6fF4xGiHEuZjtOG+T4cq0OT1Hzx0o5ukd37ZkwAvv/TgC035pbfSlDe9I7ia/cHsDzwdqrKOZK33+sO/Lz+eve35fM5675kzj970jmdxFLChRIycUNDqmE5pzKtL3nobvnjtTdmbZ06pvj5PylPzbpgPgNNdX99J2SIgJ8y/43w4ZIPTeOgVx5K9PLybi2XqOmmVi0cnJ3uZVkqP/BKp8AmWcw+Uidcoj/rvnN3AVPTeJQELbPgeLsrAF5bPoH9C/2voSPupWKTjvQR9HK1VSyEWNpFx4MK2im1rEF3rj78xgVxnxto2yNrH01BMIMUEUkwgWzuB1C1Zs4iOVN3o+gSyNEVDBiaNdCLmlcvsm7kSLxNBJ0EnQafW0snIu7SqI6JIXadTaYqKVE1ufh5AspkKCCGGHQg6CToJOrWYToppYQUmHFr36ZSbIiNZNhvpNJeiTUiplqZzVdBJ0EnQqbV0Mi2i5HMnAytdp1NtCjaVRjrZm9mNpE/lLCK2oJOgk6BTW+lEcF5QIUlINc2O06kyBeJlGI10wtN1tvD4DNtuKugk6CTo1Fo6ScWtZAPputx1OknVDX4FN9932gR0upot5rYTOKIOVRTHi+L4FlaiSlVduazBxEnraCXqjhFEan5UZ4dY0VyHu9eeLaAkiuNFcXwbkVTUlUsE6djoLpJKIySky821TWoiU01P/Vh2xCpOrOLEKq6lqzgprygnxfsuitnt53NbU2BBKjev4mbJNfQ81UvXiSnoJOgk6NRaOmmWgvPbxrKkd51OhSkY6GTojXQKOJ9m08DebEJf0EnQSdCpzXRS8wmHpqg/AJ3UvHpAMZrnTmYS8ThlaqqFYmUn6CTo1GY6qZoly0ghpPt0yk0hSDWb94lxCPPkayWRp4Er6CToJOjUWjrpFsaWrCDTULpOp9oUVWm+K26ooCalzsrNZoJOgk6CTq2lk1m86gHLIcXoOp1KUzRESPNbK6k722TqxreXqiTKCERtk6htamEhAanKgiQJGZLU0UKCHSMUpfllFX2t+1m2Um+IzQSURG2TqG1qI5LKsiATEd3sLpJKI2DKZzaXW2Jys4qwqVO6+sl2RkkAODFbs+TdkoUu2FXtBfbC1Mm5ViXuUYm1D2wNhsXsqau25pjsy6p7Ew+9p5L34JVe3g51A16s9I4L+8Z3DX/SSs9AqoQkDMs9+XFrPUihwWINqoaw7ONwbEiQpMj5jn75dntvXmzTvRf0+tdoV5z/7N36SvJDIC6rDnf5n8+jyftPn96fjSbHF6OT0fnl6dHZePLr6dnoQacsIqDHfqvh/9ZFwzl0oCFVJMdkeEps3VWYRg1Hmsq26urMpNghTJlqtuFiupsx81u7fxuPLiYfj86P3o92NxvsL6nP3aNpyuJ8fLhPVdWw5HKz0kfuBbi9GlJQPewVo8r0SWV7i8aWwguI4Obaj43kO+v5xp/daMrPNbY8v/vfAuAWnrXjhsP7MBi2s/OJWwrf4ZZCJ0bEFk/wiYVlS5KRrOPuTvBLIyB6WvMr6H5I0yCLp/M4uxEQFhB+aQizW2B1GcUPmCGA/AwgN69DGnFwaJL16+96+Yq6d+S6RbrtbjPeT5LZIO8st+nx5UlTd6CtYikYyYbe6SGjMAJLzU/25aupZ9qGos1mmhgyxJAhhgwxZPyQQ8YJ81n6/QYN2cJqzttDt+Nr5aBRGyHnb1Q2vSStLexp5K4kLViLQUMMGs8aNCAfPNAXJTycJ8gtuunXxoop9ZMnPyzJ2x8WqTBIo4F9/eiHHrWmS4KOo3DKveqRx+8ElXQZg4gf9qnHPfd9bRio3fTtYWDH0sjNr9V3jAdGJdQrWvw8uvh4Oh6ffjqfnIzOT0cn/acyWrMkDfB22GsFlSGTZC+s34vRo4uLTxeHIBpsUJFmNr/tZJjuyptHPlmtmEC0QLSopnmJahoxk3/e3Xigl57vQ6/JRqdnyWAEgWGkeiT66sur/wLr8RDJk4AAAA==",
    "case-blueharbor/gcp/audit-data-access.json": "...",
    "case-blueharbor/aws/cloudtrail-lookup-events.json": "H4sIAAAAAAAC/+2dW3PbRrKA3/MrWMrLbq1HmvsF5+Xguqu1FTuWk+zZ463UYGYgs0IROiRlr+PKfz8DiBIlgRRhmZQBL5QKLRHXRnd/6Jnp6fn03Wh0kL5308X8IBj9r/9rNPpUf15/f2z9hgOotXbKCGALYQHNNQGaQwswp5zBnOWF4wfP7hz4gz531aEvxvNFdGl+c/4SN3u8dtq+nE4+VjssZpdutSU0xs3nz93HqwuHz49DkP4jPHn1IgU/vHwDXqfhi3tXejO+uhKGmAOoAMJvIA4wDSD7C4QBhPf2Py0vZ6Y+Yk4O9bn+vZzqD/NDU56vdvxp7mbTpQTz9wbM3EU5W4ynZ7dlmNcnqp/cv26+jiflpX0z0+NJfbHqBJ/eHrjq95/dbD4up28PgrcH6BDBtwfP3h5c+isdW791vPjot/h9Fx8vXL3PcXhS3Ue928VsPDXjCz05tvW28DgJl8/l9Of4dfrq5es3qN5Tz66u4P8NvFzBWJ8HAfI/2P8Q/xNUlzy6I9XVgcaUl5XGr27w1hHXm681c3UH61RzI1Kl/nq3e9f549nyYVRaq3dYo7Z/1qdxK2VdnemetlY73VzslrFd3fOH+Wt3dv3ML+fA6fkCXD2nK+0dvwqtnXnBroRW8pBVqoGHmNzIEp75a1w90w9zYCbjI3yIxCGFo3N7FJWL0pQz9z06JOxQji613wpH5fxoMp5e/vt7figP6x31zLz7/t+S/8rpaKKnZ0cXHxfvyun35BDhQzbyAp3rqT26L8LM/d+lmy9e6ZkXcuEtyN/I9HIyqTfNL8rp3KUTd1658O0t9UHHSX3XkKo4YrHgIeM0ScLVg7veoYV7L2/mym/9UZXb3ujy2mLDD/PwYhzryaTe3Qukz+p7q31hddTMeVse+6/Ch22uPnusF+6snH2sdzm5OaW3pYPa6f54tpFb3CBOmGVAMMIBZRgDnSMIiBKIM0GlkGwDt04X5cWL8uzsns+vuFXoyXwv4CIBRVvAZSrILCrIDADrGsAq9a0H2CatrQHZLevrK8jui7AOZN5YptciX6t79ZSCGznvqr/eeJRPLt07PcvLGai/uFJNWybGGUIJyTCFIUtCJBtMbIOO+0yskfCVoVjtMJuVs7i0y8vXZp+46djZ1eYT/50/pt6j8tFg9DnuNhrPR9NyMdKXXvOz8e/Ojhbl6MLNinJ2HoxuqfCWFYxyZ7Q/lz9yNF4CA+R67g++KCdj83HkH1H5YT5avHObTqHNonaC7ehnChWIawVy4bz+bI78O01o4AqmCwslx3gT+v/qFpW2Vlh72sCVBAhuC1wX8wH83QM/ghsi1/vqWkP8htH1lfvrBdlBGItSglOFEYpjmtIIN5DdxuV7GcYqaxByuQCISQNoJaLOZQ6EYppS66ikeAPLYi/rwt3A6dGh7M/pD8nL1zde8zz9n3YwwwHEW2Dm3bwNzPxB1r/r5/q8SySLn8fpz6c/8hgvH83nkGwl0qMx1tRLE2S3L7OVYpXC1lOsoac1FLtnbn1l2DoxNsSv214Za+j26ZYu7/259Q01X+jF5XwZ2C3G793W95bfbGpxEv//VePCXYwQfjaqND9aqnwUnvj7/aMBXUUTqEIuUArTiEjegG4bNnUxTm7RecA5YtoH/8Q5BqgzHGhcKOCQxbZgRUEkeaDT88Z6Ht3v+VjmooDAgbk9Ym6lsMcz966x9bnf854UuyPu2niSKpFhDBOZJjgiWDS7AFoAoJfxpEQcKasgoEJr/+Es0EwbAGWuFRcOG2YeIttioc07ZysnfVW13MfuKzBu6+jOwLhuMW7TuE5rxq0xu17TbpM8e+YekyGlTCZZSKI0Ec2uzzZ46CX3CKJcWiIByY2PVYkTQMuiAF5gp5jUyDL6APcqRT056GBAh2CuT6CrFPZloKvtrM9kWwmwg26/kHGZJCgRqWBRmKEGrtp4dT9HrzlRhFnPYQUpoLDAlWAMOMJyrDmW0BT7y7p5PLDQbvJuBl49Ea/QkHCz24SbBDPBM4wkxyxGgq1pWW737A4ia19jyyt73TKwPCfBLXV91oDyvUPbDyRTI2AuGANEFNLrivrXC5UYGKck1c4Rw/c+kPxIFiMVULmboeQBxk8B41phwxjy/saQI8xgjCJEUZRS3wxukLmNt/cymMyNIFBxAhgxGNDq5aOxgEBSpRkj0gm7KYU7nM8vz93rcuJ+GS/e/eLyh3H28GhyC3DBgMkA4d2A66aTxHvgf9/KGfMPb/JxMTbzw6pZduaPeD82bgmBu6fbP+duPdONvFve2vzwrCzPJq66xeDRwjUx9WWnun79vpqV7/3vs013vJbPXpP3+Kxri7Ng5m3u6GRcBRW/zMbe5SsjPPqCW30YwEvDezyAN3rKI0HsIXroMcvgGgxXzCVHS+C2ZfBD/YvVww63vjrXquTqtP6XU3/vXsAdWZW9nOkqTDt1ppza6jYJh/Bz+jpFiglNWRyrNKY8Shq8b4PFfg5f04Jjy3zzghWF9pJ5GXWB/Z9MwCqp37HcbY5br+Lkulv6UVHr6fF1kl8S/hBC/4O2kR+9qfrOUADJTroPrH9aQNvz8bRLEevffwyjF/88pSH5MfnJtxJ//JyIdSXSoyLWNTppvgduX+MBWq6U9ei+g3tW9kWI5OsIeVL+Pp5M9BHzGPzTiTbj6aKcv/uv0fF04SYj/8Xo5enoHyMEf0XsV/HnUXhx4cHt8ufjxRGH7NDrm43+9Pxvb05ePBtNxr+50V+d+a3882hpFEc+/uWjU13o2fjmgF1FqhgREkYxjjGJ0zBtdnu28e+eTtohUEvDAZOQAEpJAbSXBjhoKOeKCWs3ZTsmbm5m49zVvjx/Sm4RFmCx01k7A7/2zq9aaV88V+eu0Q0Yu5tkQ1msmBQJzTAjMFwzz2a7t/czyUbkiBmngM0xBFRjC7RQEliplXTEGWPy/YzePJZhiAcUD7FXP9hVK+srjtt8q8BKZMS4SETGFGIkamYFtvHrXgILI8oNyiEopCw8ibEBGlkOKMZKCSaRk3Bzg/HELWZjk+iFfrpBDvQGqQD5l7jaQq3zcjpelB4JZ8NYRwfGOlZ6Ww+wTepa34hcWd4XoWw5GPANoQzFIcJcpBHBUYxYc8JcG4/vJcoMxrkrjAIF0RTQggmguTc+wqWGBDJGnOpQ7FV5Aw4YG2KvHsRe18oaYq+dA4vHqahARTIBEcmamclt/LqfjUVisPQfAOkCAWqdt0Umqx49CYXVyiJt9pGZ/FhcQRUwtZu05IFXe+dVra2vl5P8zeIqS0TKJEeYJQqtmUDWxqt7iSuWI2KpgcCyXHjBcAE0yR3wjWEpCOKESNSl+EpWA/tkGFfsBa+Wyhriq903CGWSRGmIUpr4/6K0WUGlhV/3c+aX0QyxwgJUOAuo9G1eDXkOcm1sQXPLLX0gGSIu54twan+qspqfElpYBHhrKUA3QKsT0KqVtWEw0bVK3b1lZQO47qZxYU5FlEGcpAomBDbngLXw7342DC0l3PnGLkZIAoq9TFqoHChNrYTaEUR1tyItzAM8RFp9gValrCHS2jmwlArjOImoSolkWaaaTcMWft3PUURc9d1BAQpXGECRUUCbQoAcIlNIZHWRi+70ZImAke0Jp0NPVjd4tdTW0JO1c1yJWJGIkihKKCOwWeWtjVf3EleUW20wJADnTleCIaCZYSDnSjiMKXIUbkk2DSd6dj5/0qwHUaUuQjpkPfQs62Gpty/OerhrekPaw93M+QjiKEuTSEHFY9WEWRuf72faA2KUFogD7OoCyIT6VjDzQaW1RlmuNetWt7yoRqagGhqL/Qi+amUNjcWdA0umHFLGJcyUJBFpTlJs49e9BJY1oqASSpBTZQEtclV121EAEYWEMS1ygbsYfUH/IpdD9NW76KvW2xB97TUpgkjOIJUxRJzDKG7ArI3P97OrXrJqCjsHHBtSVUKHoDJqQCilOXOykBJ2a94iYgHD3imGeYt9CsSWShvmLe4TY55eIQ+VhxmOUsaaJdzaeHs/YzJOCKbKAMytF4yI4mpeuSZKUIuqohiiezEZCwgPMB5isp7FZEu9DTHZPmGWqYSmJPPhGMlwuGYB2DY+30uYCaN9m9lRgJjNq3rmxgvmf/PPGRIuqRJEdWc0kgWY7WztliEO23scVmtrGI3cw2hkFhIeo4hDFNFm6Zs2Xt3PIo3YWguZBhJLCag0DOicCsAtxEoggwxHW2Kv4+l8oafGPSm2EA3IttjLGTxgqxPYqrW1HlsNJT0Qat1Y2oCvuwsWJCohPGGcxFEG42aN2TZe3s9pQcoJiyTykvjgkSosgUayALkuqKKF0wKrjvWA0eolvnWJ0qEHrFMIWypt6AHb62QhTjHFPpwimMgYiTXLLW/39l5irAojkZEEeBBDQJ12VX5IXq0ASHOHiNPKblpu2T/ZcuJelGd3nX9X1bFr44e+5bG1rP/4bDqeDrRqQatHQ6jWw4Z8iDWPf91yybfM5T8XP5/uP4hqjeLL2jGuV7a7ky4hME04Z4xJLDyjGmBq47+tazov7+zUK/R42pEUe868KIoBipHyUkEOvEQGFIxr5olsmDVdm8zovQVv758fJjN2JL6qlTVMZtxLYBWlcaJoREOexZnia5LtW/h3LwMrhRC0hCKAFSsAhVoAXRAMlC2QY4JQlRddWIMELZeCwHBYg2RYg+QJ1yBZGd6wBsk3swYJjzPOsyhNZMpQvKYMYxss9nMNkoL5VxYRCnAofCTOpARaYwIcpZQJ6QTVuGPjGbAqrtyG/cN4RgeC1ZW2hvGMvU1PqKrwy8gHpZ5jiWxWZWzj5f1MhYPM5DSHwDDuA3GCq44ExAHDUBNKcgIV6xy+kAqYCCgb8NUHfF1ra8DX3lYg8dCiWRpCHiHJcLMURxsv72cpDqGo04IAha0EVFsEtKMEGM9qSrTlDJGOJL/VjkBJANmQ/NYTbNXaGpLfdo6rmCmaZhBmSRhnYXPiQRuv7ieuIM8LKTAQWhFAC8d8GGkh8OEj1Rw6bpHs2MSD2hGgCPBQiqNXEw9WehsmHuwVZkRKxBglIedSyawJsxY+38+6QlrBnOQFgAhWNUa0AloWOYCC+KaxgMIi053YSwa4RX3sIfbqRuy11NYQe+08c5chmaQw44onOObNMrNtvLqfa+66XBQGFsAp7gCFlHvBpAVYQO0kL6BArHuxV9VxEjAyxF49i72Wehtir/2u/hYjRDOsMhjHKlqz8m4Ln+/npE9NWVFwBKDNq8XsqgUMpIIgVwpzRRGmuelMGbTaHaq1qOlQBq0PwddSWUMZtN2PM1JFJckoSmDMEkia0z5b+HUvgQWdFUIVFCimbVVLxIeVCBngLObYGM0szrs3zlgXBGRyGGfsCbdqbQ3jjPvCF4MsSlNGMsnDDK5ZDa6Nl/cSXyi3EgkCQQEJBdSgakV0pIFUUngxCVHcdmnaJ6wmQVMaQD5M++wTwpZKG6Z97rUPLEQRT5VCaYYzGjZnV7Xx9p4W30Ck8DYMclQJpp0BmhUYMCqosEYxWujOYayqQgMHjPULY7XSBoztE2OEZhCFEoYxE5JQvKYIx3Zv72c0JoxURBUAC5cDCnFVmBIhgPOCMYYsLDYudfm1MEYDUqVBDhjrE8aWShswtte5ojwjvkkZZkkSwRA2EyjaeHs/54paYoQuFJA293zOqzLh0osoNOYUWuhbzvnmSe5XPayvysnYfHxSjuGAoKEjvyf8qpT16I78e1Y2gOsuuCBDGU2gTFJGE9YcfWzj3/3szFfOaWQogMSHk5Rx7gNLXQBNhYauUJ7L+VeqHgSXVWsQGaoHfc3qQSs9DNWDnrp6EE4yCYWISASxSFAzx6uN//a5elBuhCAwZ1VL10eMUudA45q+HrqUSIEl60IRDrishYDUUIRjKMLxhEU4VoY3FOH4ZopwqFRRKUOhJGY+KG0WXWqDxS4X4fCf//ruj+/+H/URPUXS2wAA",
    "case-blueharbor/aws/cloudtrail-s3-data-events.json": "H4sIAAAAAAAC/+3d227kthkA4Pt9CsN7G455+HnSVXUsjGTjhbNNixZFwaMzzXjGlcab3QR591Jjb9Z25LRAiNqoOYDtsUWNKEr8/FMUqZ9eHR0dnwe3G/10XB39Lf16dPTT4XtaEN6H7f7bME7r3TYtPSYrgo+/+LT0egrjqU8p1vuPaemntdKS/cerMKevp+n6Mvjz3Sb8slpafDWut259ZTan/pDq/Kzu/1K/eftV/+b0/PzsHGNMqjFc7cb9enuBJvMHu7kO35nR7kZktmbzcb9202ptLlcXKQ/v1y4Y53bX2/3K7S7vbsmMh3ynH5X5Yaqm/VRVJL1oerH0qsxNDtGYsnjyZj2Ou/HP43ofxjnPJ3nycPPnm329u/EHicI0fRk+3hbJN6cPi4TcTT6lxOmYtLvtPnzY3yv8z0tP076F8cHCO0fnwWFZKrC0ew8KbLGgHn7KfGZ8bS4PW/lV2jtJf3718N3PX9w7996tbz6EYioQ1ojgdxhXXFVc//X4ftJvdtejOySe2Mpcmh9327QH9w7GTcJPGXt7vT+z/wxu/3l5WuE8XNye7NcTCmbao88FfzwdtnH6tvZ+TGV8OKCarvCKrgh/UDPqi7StOUWz2+/YCVkxvlJHl/5k/j3Vt/D69k/X5iR9xOeVx/Cv6zDt35oxZTQV2nS/btlr9334ZSfunJThw3yyTujyUOB3z5bvw1w/j28TnNwW5uEn1gSjKzPuEcYrN71fXfx4/KtDkfb1aredQr8Jl2mn5gxtrzebhzk+7eatdB3rqYC26RUGqeWD0r9JRHAQ3nOGBGMEAdUUmcglwiFIYZUVxJq7JWL82XYz70Q0myk8OEc+YfPDVF+tW7PZfF7zMtXUi0Om+/c3h+P+B4whObROS+r/WElvNtaafbjYjYfy7Mze3BTWbVEVN4ubz9dNUiU6cXEzt5skj5tNI5uhIYKAYpJKWHSTY+oU5hwZYjgCHnB6BxiB89RoZ6UIuLhZ3CxuZnWTVKS4mdtNmsfNFrO6H7Tqh6ar9UAX3QyMBiGlRNbQWcvIkGGeIRwVYUpaoQ0UN4ubxc2sbtKKFjdzu8nyuClph4ELRqhqalmTRTcVcdIECOmABo0gaIEMmNROd1w746KSIRY3i5vFzaxusooVN3O7CXncBK2oboQE3PZd/Ug7XQEwCpYgolPlA0MlMlQbxIOSXhsitVDFzeJmcTOrm1BBcTO3mzyPmz0deqk1VqJltOXL7XTNIAjnE5k0bR8sJshoR1Nj3fgISgQaZXGzuFnczOQmOfSn6wrj4mYmN0ne/nTCaCNwx5imBHjXLboJlIfICSAmKEHgUqhpgrFIGAEmKOKN18XN4mZxM5+bh/50UtzM7Wam/nSNOz1o1am2rmul+KKbzluntYko2BAQJDiRIcIhiQmTUUSKXekXKm4WN/O6SSpa3MztZqb+dNIMnNCm6QfVSRCPxJtGegHYIqN8RBBpRIZGk95x4hiWWlJS3CxuFjezukkrVtzM7Wam/nTR8L5mdd+QTrChZcv3IVGjQtAKgY8cgVYp3ow6MeqsFpQpb3npFypuFjfzuskqKG7mdjNTfzoX0FI8DC3mmPXNI/e9My6Z5oAcD2S+D8kjY41ClkulhaZRcFHcLG4WN7O6CRUvbuZ2M1N/OpVUdUxjMvRUAVseZ2kjE05zgbykFgHhBhlDA/JaWwU8EsdLO724WdzM5Sb9ND6dFjczuUnz9qeruoOmwQ20GPPUZl/uF7KYiwACCa3EPD4dpyZ6CjqZIcFErCKLpZ1e3CxuZnTzZnx6cTO3m5n60yXIHisG0EhOlFzuT9fWCp0qTCphYAhESPEmZxjRaLUQYJ2mobhZ3CxuZnWTVKS4mdtNmstNJgfaD4NUCmtVL7ppnEpN+flGd8o9AgzzeKGoUBSJUy4EC8IVN4ubxc2sbtKKFjdzu5mpP513tMb9wFupVVPT5f50S0ApkbIhmQsIpHXIRKWRij5gJp0wZZxlcbO4mdlNVrHiZm43c41PF2Qg7UDZ0Dct7ZfjTYmt1pACTOJIchMzMQ+xxMhyIBJMasKH4mZxs7iZ102ooLiZ281M/em4JowpCkrgoZetWJ7XIzgarRKIOc4QUK9QwtIhpoh2mEeK/Uu+D+m0fvOnlOi3zDztPgHxzbftef/27Pwd+Q35fl2R51ycTO8d+gW5343Wl6c1us0U+vrsHTrv66/upr/Lwv0t//e1PzU1oWLkd9f+P4Y8tV+t+Hx0EwHskfqfPha5zTpVciJXgB8n4Gg3nWzW2+sPr8VKrQ4Jzei+e/1BiX8IONqY7cXJ1cf9d7vta7YidMWP0u6lM9yfpH12V08mCM49UrtXhEiMJRl0M4hm+Q5wwoPHYJGRniNQkiCDmUHaU+518CH5sijIfrx+VoB8Pv28X+/T2Wc2h00dFt4/bh/3YXo3mu0UwzgGf3Y954dwqjnTkr66+5+0iPS/FInnGE9SRMovUqZraK1u+6YburbluGZkeSxfxERLyVNbMMaIgGNAJkiKnBBKYRW4Ev6FiCQ5ABeyiPRkIomKqCLSMxIp8913HBQZCO+1UG0P9bJIzFsqpLXIW+Lmu5YDMi7FSNgSm6AyihP9UmIkrKkQooj0ZCLJHHO8FJHyi5QpRlJCDQQrpjnpaqUfiZGMMFhGiTAEhsCwFCNRxZDxUQhuqTcuvBCRFNdYgS4iPZlIKsfsKUWkfCKxvDHS0In0P59hVnc9Y7Dcg8ft/EgTFZHBfr4SLQgylinEjWdzyGCMwC9EJCKZVLqI9HQi6SLS8xQp13UkJXpoaiwINAM8IhLTSmLmAQnNAgJP8DzWlKIopQCiObP2pVxHIkoQjkur7clEYriivIj0jESCvDESI6Rpe0hhUieHpl1+ehGLOihJBBIOLAIXUoxELEMRG8+Jsyo+8vSi/0eRiBYlRnpCkUiFy5Xt5yhSrhip6Zq+V62QNQxCDcvjygPRcb5VXZJ5/jfvDDLROWQxAWohEu5eSowEGAiWRaSnE4nmeKptESmfSDzzTBeUMGAU44ZpVovlVpswRFCjBDLWawQkYWSEBMTMPLGak9ryl3IdCTjWBHAR6clEYhWDItIzFCnXHLmMtn0tmVBYUQbLc+/QCNwErZGfJ3yE+WlghluPIuMyUAoSwLyU+5GS3umriPRkIkGOJ1gXkfKJJPLGSJ0aOl4PQ8s7PAykXZ6dQQqOHSNIeMHnpxKmGMml9hsNwioFBLjnL0QkRvQ8+08R6clE4hUr15Geo0gs13wxneznK9ugadvB8nNSSXQ4BiuRpHPvvxap/RYtR8oCgIx8XvuFiCSkZKxc2X5CkUSOWaKLSPlEknljJOBSSs0bTpNMVCzPNE055oEFiyKPfr5nWyEDiqRWG6MarDSp+fZS7tmmYh4PXER6MpFkxUUR6RmKlOs6ktC1giSSxpwL9sh1JCLAERsRwPzAJWU0MtQYRCTlEYK1wb6cGElLhdV9kdL3v7/6+dW/AX/d6AkCngAA",
    "case-blueharbor/azure/activity-log.json": "H4sIAAAAAAAC/+2WS4/TMBCA7/0VVc/YsZ00TXLaPQCqEA+xBSQQB8eZttEmcbCdwoL470z63MKyVK2ASpuRIkuZh8fjT+P50Ov3v+HX7w+ULAowg6Q/sLKkBm7AXiygyrQh8EWWdQFU6XLwaGUNqHKTvATrUNd6CSZCwmLCxYSJJAgSzihj7P3GQ9dgpMt19UKWgA6rbVGxkEXT/hg8z5XRVk8dvXLayBl4drVeKqWbylmvyK17BjfWk6qNtA6NMQqN6edfIXt7fLBlrO/rbA1Y3RgF46wN5tkmtcrkdWtoPbYWwlGIQCE+CgluibeJ8NToprZeKc01uLyaEZs78GqjF3kGxnp/TjSdb52lteDspqRYe9fYO2v5ROYFZPcVaG2xd+q5c/Vr+NTgrd6Oqoocb3tcX2YZnqrdcMDjiA455YxR4e+2KcHN9bJkr15eTfaDF8hM0aoeG6NNq1tqfsFvlHElUl8SBvGUBDJKSRqqIckC8KdC8pSp+AAM+YSPkqGfBCPKeHAUhu8g9drbst5ng8thuP3s9N+x2iWUzsnW/gCGrhqlALL7MdoZHU2SoAgRFYzdydGb32E0rqbalMu7lMW/wClMeIxE0YhFHU4PFadMVpLqa4nGFykmPZcm1Yaun8hDQWIs4RGNouGJICldTfPZcTzt+Z4pVtskIT07wsJz71c4grUti/Iw7PpV9/ydhBObCJawIPFDxCnucOpwOhEnHKUCP/FHdMSHHU7dNHXkNMXW01TAqR+Jbprqpqm/1a/w7RP4AtJ45Hf96gH0q97H3g+16lLkHhUAAA==",
}

EVIDENCIA = {}   # nome do arquivo -> bytes
ORIGEM = {}      # nome do arquivo -> "github" ou "embutida"

for nome in HASH_ESPERADO:
    try:
        # timeout curto: no Colab sem rede (ou num teste offline) não queremos travar
        with urllib.request.urlopen(BASE_RAW + nome, timeout=15) as r:
            dados = r.read()
        if hashlib.sha256(dados).hexdigest() != HASH_ESPERADO[nome]:
            # O arquivo mudou no GitHub: não é a evidência que recebemos. Não usamos.
            raise ValueError("hash diferente do esperado")
        ORIGEM[nome] = "github"
    except Exception as erro:
        dados = gzip.decompress(base64.b64decode(_AMOSTRA[nome]))
        ORIGEM[nome] = f"embutida ({type(erro).__name__})"
    EVIDENCIA[nome] = dados

# Recibo de custódia: confere TODOS os hashes de novo, depois de carregados.
for nome, dados in EVIDENCIA.items():
    ok = hashlib.sha256(dados).hexdigest() == HASH_ESPERADO[nome]
    print(f"{'OK ' if ok else 'FALHA'}  {nome:55s} origem: {ORIGEM[nome]}")


def carregar_json(nome):
    """Devolve o JSON já convertido em objetos Python (list/dict)."""
    return json.loads(EVIDENCIA[nome].decode("utf-8"))


def carregar_jsonl(nome):
    """JSONL = um JSON por linha. Devolve uma lista de dicionários."""
    return [json.loads(l) for l in EVIDENCIA[nome].decode("utf-8").splitlines() if l.strip()]


def carregar_csv(nome):
    """CSV -> lista de dicionários (cabeçalho como chaves)."""
    import csv, io
    return list(csv.DictReader(io.StringIO(EVIDENCIA[nome].decode("utf-8"))))

In [ ]:
# ------------------------------------------------------------
# "MINI-JQ": a função pega() faz o papel do caminho com pontos do jq.
#   jq '.protoPayload.status.code // 0'   ==   pega(e, "protoPayload.status.code", 0)
# ------------------------------------------------------------
def pega(obj, caminho, padrao=None):
    for parte in caminho.split("."):                      # anda campo por campo
        obj = obj.get(parte) if isinstance(obj, dict) else None   # campo ausente vira None
    return padrao if obj in (None, {}, "") else obj       # o operador "//" do jq: valor padrão


def ua_curto(e):
    """Primeira palavra do user agent, como o split(" ")[0] do jq."""
    return (pega(e, "protoPayload.requestMetadata.callerSuppliedUserAgent", "") or "").split(" ")[0]


def tsv(*campos):
    """Imprime uma linha separada por tabulação, como o @tsv do jq."""
    print("\t".join(str(c) for c in campos))

In [ ]:
# Os dois arquivos do GCP juntos: é o "jq ... audit-activity.json audit-data-access.json"
GCP_ATIVIDADE = carregar_json("case-blueharbor/gcp/audit-activity.json")
GCP_DADOS = carregar_json("case-blueharbor/gcp/audit-data-access.json")
GCP = GCP_ATIVIDADE + GCP_DADOS                               # o "jq -s 'add'"
GCP.sort(key=lambda e: e["timestamp"])                        # o "sort_by(.timestamp)"
# AWS: cada evento do lookup-events traz o detalhe numa STRING JSON -> json.loads é o "fromjson"
AWS_MGMT = [json.loads(ev["CloudTrailEvent"]) for ev in carregar_json("case-blueharbor/aws/cloudtrail-lookup-events.json")["Events"]]
AWS_S3 = carregar_json("case-blueharbor/aws/cloudtrail-s3-data-events.json")["Records"]
AZURE = carregar_json("case-blueharbor/azure/activity-log.json")
print(f"GCP: {len(GCP)} | AWS gerenciamento: {len(AWS_MGMT)} | AWS S3 dados: {len(AWS_S3)} | Azure: {len(AZURE)}")

## 2. Os cinco filtros

Cada função recebe a lista de eventos do GCP e devolve só os acertos. Nenhuma delas sabe o IP do atacante: é a regra de hoje.

In [ ]:
import re
from collections import Counter

def f1_negadas(eventos):
    """jq: select((.protoPayload.status.code // 0) != 0)"""
    return [e for e in eventos if pega(e, "protoPayload.status.code", 0) != 0]   # 7 = PERMISSION_DENIED

ALTO_IMPACTO = r"SetIamPolicy|CreateServiceAccountKey|DeleteSink|setMetadata|instances\.(start|stop)$"
def f3_identidade_politica_logging(eventos, padrao=ALTO_IMPACTO):
    """jq: select(.protoPayload.methodName | test("SetIamPolicy|...")) — confira os nomes EXATOS no seu log!"""
    return [e for e in eventos if re.search(padrao, pega(e, "protoPayload.methodName", ""))]

def f4_origem_cliente(eventos):
    """jq: [email, ip, ua] | @tsv ... | sort | uniq -c  -> Counter faz o uniq -c"""
    return Counter((pega(e, "protoPayload.authenticationInfo.principalEmail"),
                    pega(e, "protoPayload.requestMetadata.callerIp"), ua_curto(e)) for e in eventos)

def f5_pares_novos(eventos, corte):
    """Linha de base: pares (identidade, método) vistos ANTES do corte; mostra só os novos depois dele."""
    par = lambda e: (pega(e, "protoPayload.authenticationInfo.principalEmail"), pega(e, "protoPayload.methodName"))
    vistos = {par(e) for e in eventos if e["timestamp"] < corte}       # texto ISO ordena como tempo
    janela = [e for e in eventos if e["timestamp"] >= corte]
    return janela, [e for e in janela if par(e) not in vistos]

def mostra(lista):
    for e in lista:
        tsv(e["timestamp"][:19], pega(e, "protoPayload.authenticationInfo.principalEmail"),
            pega(e, "protoPayload.methodName"), pega(e, "protoPayload.requestMetadata.callerIp"))

In [ ]:
print("=== Filtro 2: chamadas negadas ===")
negadas = f1_negadas(GCP); mostra(negadas)
print(f"\n=== Filtro 3: identidade, política e logging ===")
alto = f3_identidade_politica_logging(GCP); mostra(alto)
print(f"\n=== Filtro 4: origem/cliente por identidade (uniq -c) ===")
combos = f4_origem_cliente(GCP)
for (quem, ip, ua), n in sorted(combos.items()):
    print(f"{n:5d} {quem} {ip} {ua}")
print(f"\n=== Filtro 5: pares novos em 12/set ===")
janela, novos = f5_pares_novos(GCP, "2026-09-12"); mostra(novos)

## 3. AWS e Azure com as mesmas perguntas

In [ ]:
# AWS: jq '.Events[].CloudTrailEvent | fromjson | select(.readOnly == false or .errorCode != null)'
aws_hits = sorted([d for d in AWS_MGMT if d.get("readOnly") is False or d.get("errorCode")],
                  key=lambda d: d["eventTime"])
for d in aws_hits:
    tsv(d["eventTime"], d["userIdentity"]["arn"].split("/")[-1], d["eventName"],
        d.get("errorCode") or "ok", d["sourceIPAddress"])

# Azure: jq '.[] | select(.status.value != "Succeeded")'
print()
az_hits = [z for z in AZURE if z["status"]["value"] != "Succeeded"]
for z in az_hits:
    tsv(z["eventTimestamp"], z["caller"], z["operationName"]["value"], z["status"]["value"],
        pega(z, "httpRequest.clientIpAddress"))

## 4. A tabela de filtros (calculada, não copiada)

O checkpoint exige que **toda contagem bata com o que o comando imprimiu**. Aqui ela sai do próprio código.

In [ ]:
TABELA = [
    ("GCP chamadas negadas", len(GCP), len(negadas)),
    ("GCP identidade, política e logging", len(GCP), len(alto)),
    ("GCP combinações identidade/origem/cliente", len(GCP), len(combos)),
    ("GCP pares (identidade, método) novos em 12/set", len(janela), len(novos)),
    ("AWS escritas ou erros", len(AWS_MGMT), len(aws_hits)),
    ("Azure não bem-sucedidos", len(AZURE), len(az_hits)),
]
print("| Filtro | Antes | Depois |\n|---|---|---|")
for nome, antes, depois in TABELA:
    print(f"| {nome} | {antes} | {depois} |")
maior = max(TABELA, key=lambda t: t[1] - t[2])
print(f"\nMaior queda: {maior[0]} ({maior[1]} -> {maior[2]})")

## 5. 🎮 Quiz: lista curta ou excluído?

Para cada evento, **S** = vai para a lista curta, **N** = revisado e excluído. O porquê sempre cita evidência da própria entrada.

In [ ]:
MODO_JOGO = True

QUIZ = [
    ("GCP 01:47:03 sam.reyes storage.buckets.list NEGADO, 198.51.100.23, python-client", "S",
     "Negado, origem nova, cliente de script, sábado de madrugada."),
    ("GCP 10/set 15:42 dana.okafor SetIamPolicy (adiciona lee.chen a bigquery.dataViewer), escritório, navegador", "N",
     "Origem do escritório, navegador, papel de leitura do BigQuery para um analista: explicado pela própria entrada."),
    ("GCP 01:58:31 sam.reyes CreateServiceAccountKey para reporting-sa", "S",
     "Chave emitida para uma identidade mais poderosa."),
    ("GCP 00:28 scheduler-sa instances.start report-runner-1, origem private, Google-Cloud-Scheduler", "N",
     "Rotina noturna do job de relatório, todo dia no mesmo horário."),
    ("GCP 02:03:12 reporting-sa instances.start, 198.51.100.23", "S",
     "Conta de serviço chamando de fora, segundo start da VM depois do agendador já tê-la parado."),
    ("GCP 03:06:15 reporting-sa DeleteSink audit-to-bq NEGADO", "S",
     "Tentativa de cortar a trilha de auditoria (T1562.008). Até falhando é barulhento."),
    ("AWS 00:58 AssumeRoleWithWebIdentity de 192.0.2.150 (NAT do GCP)", "N",
     "Mirror noturno; endereço NAT da própria Blue Harbor; todo dia."),
    ("AWS 02:23:41 svc-reporting StopLogging AccessDenied", "S",
     "Tentativa de desligar a única trail da conta."),
    ("AWS dana-admin ConsoleLogin do escritório", "N",
     "Admin, escritório, horário: linha de base."),
    ("Azure 02:44:10 sam.reyes listKeys Failed, 198.51.100.23", "S",
     "Mesmo movimento (emitir credencial) na terceira nuvem, barrado."),
]
acertos = 0
for i, (ev, certo, porque) in enumerate(QUIZ, 1):
    print(f"\n[{i:02d}/{len(QUIZ)}] {ev}")
    if MODO_JOGO:
        r = ""
        while r not in ("S", "N"):
            r = input("   Lista curta? (S/N): ").strip().upper()
        acertos += (r == certo)
        print("   ✅" if r == certo else "   ❌")
    print(f"   Gabarito: {'lista curta' if certo == 'S' else 'revisado e excluído'} — {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}")

## 6. ➕ Operação Isca: o funil nos logs da Lia

Mesmos filtros, sem saber o IP. Linha de base: 30/set e 1º/out. Janela: 2/out (UTC).

In [ ]:
# ------------------------------------------------------------
# DADOS DA OPERAÇÃO ISCA (fictícios) — mesmo esquema offline-first
# ------------------------------------------------------------
import base64, gzip, hashlib, json, urllib.request

BASE_ISCA = "https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/"
_ISCA = {
    "dia-77/lia-auditoria.json": "H4sIAAAAAAAC/+2ZbW/bNhDH3/tTCAYGbEBFkbLs2OqbBo1XeFu8oE42YMMQUNJZYUOJKkl5cYZ+951kee5arE68uIgTvbNPfPjf3U/ikfy94/zVcZyuVOmUZ9ANnW6h1TuIrfEMaM35bSldKbiHLYwXS1UmvEyEJalSqQReCENilX3jf59wyy95HIMx3RfVmDiQVWd8KRVPcOBqHqfLS3sFuRUxt0Llk3yu1o+qDiKPRcHlOONCVlpwYjdS9tW/pBDBM5KiaSFiwAlVmVsCNzwrJHSrkT68qKfKwF6pZO3WSi+pPSCRSN+XoJdk4ZMfVDRbjUUmOY5q0dBdjaABWxl7CpZX3m2UxlxK0JNiFS+x4BZWXdaPZmVRSAHJBQ54nKLDa2+wZZ6IDC3KYyT4WK8Go0odw5ZEVEoMoDlGUcZqdQmmAHmlPMsjCWhXqgDNE2UaN4zltjSV+g+dZrauRQ1oz2oXfOoPXDpye/ScjkLmhzQglNLfUF3VuiWkJWRDCOuFAQ37bB+E8NiKhbDLB8MjgUKqpRuLBwdEl3nFRgOGIRcFhh2av/uho57YjUohkx2okGoVMrQrjCE21hhEF5AQ5jWhMN4nBO7CRz9kNPT9Q/mCJLDA1FvQuXpVew74xPxPGt6A3QEFNsKoEZ8M6R1oOFW3Qkru9Ql1vv1V5In60zjTc4dRQl86aBgEL52bQfDdY2ZlEFI/rBQ/C1ZQcyrytCLlp9XPBpJf0CCMReM4t1qAaan5MjU+DYN++4VpvzB3YCVgIe219Wxbz/4HIcOQ9sNeS0hLyGeEMOpS1u6JW0K2ELLPPXFLyJMg5MBWmV1rVmOV5imQqIyvMZhE4rbmfgXqkOB7xCglg8Edcr8KhovRcIslCsndGNvk1vMJC3pV4ZreimKHCvWzHDdSVFJ1OmqUYbINulsNczZ+ezqZzSY/Ty9PxtPJ+GQ15xYy/HPaq8hgT/68BDcsE56dKSniZUvEdiKC53IqUi07PMkE7m9ZfQzSLBvHqyXIPA1Y7pb2wfM5ON2S9h9heZiZX59erPH17l9s7cbOUUj9A7mUeThwXmvYXLts0GnJacj5Z/5CYRG+CUC+ZyXeNb6/3lHcgxHjNAkivz8fDIHFlI+io6Q374MfB3wQDRM6Z+uoY6fzZVELu5iN316eHk+P39xr3RyFvSe8C2vCZD7Ffe2d59W9N+LX2Xgkaf76u8jH83Lvc39Zs8/8kPVb9lv2Hx/771RkPFzyNdYSrlVuaiqza7PChZtCabsr8sM9bZMevOZpef/CZfhrlc/FR3fhJyABizqRXx9oHYfKjVejpTSGueAaY/3+ax4gVMfRm8Wg80fnb1Tsfa3qKQAA",
}
ISCA = {}
for nome, b64 in _ISCA.items():
    embutido = gzip.decompress(base64.b64decode(b64))            # a cópia que veio com o notebook
    try:
        with urllib.request.urlopen(BASE_ISCA + nome, timeout=15) as r:
            baixado = r.read()
        # só aceita o arquivo da rede se for idêntico ao embutido (mesmo SHA-256)
        ISCA[nome] = baixado if hashlib.sha256(baixado).digest() == hashlib.sha256(embutido).digest() else embutido
        origem = "github" if ISCA[nome] is baixado else "embutida (hash diferente)"
    except Exception as erro:
        ISCA[nome], origem = embutido, f"embutida ({type(erro).__name__})"
    print(f"{nome:35s} sha256={hashlib.sha256(ISCA[nome]).hexdigest()[:16]}...  origem: {origem}")

In [ ]:
from datetime import datetime, timedelta
LIA = json.loads(ISCA["dia-77/lia-auditoria.json"])
brt = lambda s: (datetime.strptime(s[:19], "%Y-%m-%dT%H:%M:%S") - timedelta(hours=3)).strftime("%d/%m %H:%M BRT")

l_neg = f1_negadas(LIA)
l_alto = f3_identidade_politica_logging(LIA, r"SetIamPolicy|CreateServiceAccountKey|DeleteSink")
l_combos = f4_origem_cliente(LIA)
l_janela, l_novos = f5_pares_novos(LIA, "2026-10-02")

print("| Filtro (Lia) | Antes | Depois |\n|---|---|---|")
for nome, a, d in (("Chamadas negadas", len(LIA), len(l_neg)),
                   ("Identidade, política e logging", len(LIA), len(l_alto)),
                   ("Combinações identidade/origem/cliente", len(LIA), len(l_combos)),
                   ("Pares novos em 2/out", len(l_janela), len(l_novos))):
    print(f"| {nome} | {a} | {d} |")

print("\nCombinações (uniq -c):")
for (q, ip, ua), n in sorted(l_combos.items()):
    print(f"{n:3d} {q.split('@')[0]:12s} {ip:15s} {ua}")

fora = [e for e in l_janela if e not in l_novos]
print("\nNa janela, mas NÃO são pares novos (o filtro 5 perde):")
mostra(fora)

> 💡 Os dois `InsertJob` que o filtro de linha de base perde são **a extração dos dados**. Quem os pega é o filtro 4: `lia-bot` nunca tinha chamado de `198.51.100.66` com `google-api-python-client`. Um filtro só não basta.

### A lista curta para a Marta

In [ ]:
#@title Lista curta da Marta { display-mode: "form" }
autor = "Analista júnior, Vigia"  #@param {type:"string"}

MOTIVO = {   # motivo COMPORTAMENTAL, nunca rótulo de severidade
    "list": "storage.buckets.list negado; conta externa, origem nova, cliente de script, 00h BRT",
    "GetIamPolicy": "negado; mesmo padrão de sondagem",
    "ListServiceAccounts": "reconhecimento de contas de serviço logo após as negações",
    "ListServiceAccountKeys": "olhou as chaves da lia-bot antes de criar uma",
    "CreateServiceAccountKey": "chave emitida por conta externa para a identidade da Lia (T1098.001)",
    "InsertJob": "lia-bot consultando o cadastro de fora da nuvem, com chave nova",
    "DeleteSink": "negado; tentativa de cortar a cópia dos logs de auditoria (T1562.008)",
}
linhas = ["# Lista curta — projeto serraazul-lia (2/out/2026)", f"Autor: {autor}", "",
          "| Hora BRT | Hora UTC | Identidade | Método | Motivo |", "|---|---|---|---|---|"]
for e in l_janela:
    m = pega(e, "protoPayload.methodName").split(".")[-1]
    linhas.append(f"| {brt(e['timestamp'])} | {e['timestamp'][:19]}Z | "
                  f"{pega(e, 'protoPayload.authenticationInfo.principalEmail').split('@')[0]} | {m} | {MOTIVO.get(m, '(escreva)')} |")
linhas += ["", "## Revisado e excluído",
           "- 30/set 15:10 UTC, deploy-ci UpdateService: origem private, cliente cloud-build, horário comercial (deploy).",
           "- 30/set 16:02–16:41 UTC, dev.externo GetService/ListLogEntries: navegador, 192.0.2.80, à tarde (suporte rotineiro).",
           "", "## Fonte ausente",
           "- Log de login da conta dev.externo@atendemais.example (provedor de identidade da Atende+): diria como a conta foi autenticada à 00:05 BRT."]
texto = "\n".join(linhas) + "\n"
from pathlib import Path
Path("notas").mkdir(exist_ok=True); Path("notas/dia-78-lia-shortlist.md").write_text(texto, encoding="utf-8")
print(texto)
try:
    from google.colab import files
    files.download("notas/dia-78-lia-shortlist.md")
except ImportError:
    print("(Fora do Colab: arquivo em notas/)")

## ✅ Antes de fechar o Dia 78

- [ ] Tabela de filtros do Blue Harbor com as linhas dos filtros 3 e 4 preenchidas
- [ ] Lista curta (≤ 15) com motivo comportamental, e "revisado e excluído" com o `SetIamPolicy` de Dana
- [ ] Fonte ausente: log de sign-in do Entra ID
- [ ] Lista curta da Marta (Lia) em BRT

**Amanhã (Dia 79):** a pergunta muda de *o que* o ator fez para *por que ele pôde*. IAM: quem tem o quê.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>